In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO
#custom dataset class
import os
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np


# 1. Custom Dataset Class
class UnderwaterImagery(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None, mask_transform=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.transform = transform
        self.mask_transform = mask_transform
        self.images = sorted(os.listdir(images_dir))
        self.masks = sorted(os.listdir(masks_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # Load image
        img_path = os.path.join(self.images_dir, self.images[idx])
        image = Image.open(img_path).convert('RGB')

        # Load mask
        mask_path = os.path.join(self.masks_dir, self.masks[idx])
        mask = Image.open(mask_path).convert('L')

        if self.transform:
            image = self.transform(image)

        if self.mask_transform:
            mask = self.mask_transform(mask)

        # Convert mask to tensor
        mask = torch.tensor(np.array(mask), dtype=torch.long)

        return image, mask

In [ ]:
# 2. Transforms
import torchvision.transforms as transforms
import os

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

mask_transform = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST)
])

#FIXED PATHS, WE NEED TO SEPARATE FOR TRAIN AND TEST
images_dir = os.path.join(path, "dataset", "images")
masks_dir = os.path.join(path, "dataset", "masks")

In [ ]:
# 3. Create Datasets (FIXED PATHS)

from torch.utils.data import random_split

dataset = UnderwaterImagery(images_dir, masks_dir, transform=transform, mask_transform=mask_transform)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [ ]:
# 5. Display Images and Masks
def show_samples(dataset, num_samples=4):
    fig, axes = plt.subplots(num_samples, 2, figsize=(8, 4*num_samples))

    class_names = ['Background (waterbody)', 'Human divers', 'Aquatic plants and sea-grass', 'Wrecks and ruins', 'Robots (AUVs/ROVs/instruments)',"Reefs and invertebrates","Fish and vertebrates","Sea-floor and rocks"]

    for i in range(num_samples):
        image, mask = dataset[i]

        # Denormalize image
        img = image.permute(1, 2, 0).numpy()
        img = img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]
        img = np.clip(img, 0, 1)

        # Show image
        axes[i, 0].imshow(img)
        axes[i, 0].set_title('Image')
        axes[i, 0].axis('off')

        # Show mask
        axes[i, 1].imshow(mask.numpy(), cmap='tab10', vmin=0, vmax=4)
        axes[i, 1].set_title('Mask')
        axes[i, 1].axis('off')

    plt.tight_layout()
    plt.show()

show_samples(train_dataset)

In [ ]:
# TO DO
!pip install segmentation-models-pytorch


In [ ]:
import segmentation_models_pytorch as smp

# Load pretrained U-Net with EfficientNet-b1 encoder
model = smp.Unet(
    encoder_name="efficientnet-b1",     # Backbone
    encoder_weights="imagenet",          # Pretrained on ImageNet
    in_channels=3,                       # RGB input
    classes=8                            # 5 classes: Background, Tail, Body, Legs, Head
)

# Move to device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(model)

In [ ]:
# TO DO
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

# Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for images, masks in tqdm(dataloader, desc="Training"):
        images = images.to(device)
        masks = masks.to(device)

        # Forward
        outputs = model(images)  # [B, 5, H, W]
        loss = criterion(outputs, masks)

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


# Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    total_iou = 0

    with torch.inference_mode():
        for images, masks in tqdm(dataloader, desc="Validating"):
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)
            total_loss += loss.item()

            # Calculate IoU
            preds = outputs.argmax(dim=1)  # [B, H, W]
            iou = calculate_iou(preds, masks, num_classes=5)
            total_iou += iou

    return total_loss / len(dataloader), total_iou / len(dataloader)


# IoU Calculation
def calculate_iou(preds, masks, num_classes=5):
    iou_per_class = []

    for cls in range(num_classes):
        pred_mask = (preds == cls)
        true_mask = (masks == cls)

        intersection = (pred_mask & true_mask).sum().item()
        union = (pred_mask | true_mask).sum().item()

        if union > 0:
            iou_per_class.append(intersection / union)

    return np.mean(iou_per_class)




In [ ]:
# TO DO
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

# 1. Define Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 2. Train for 10 epochs
EPOCHS = 10
history = {'train_loss': [], 'val_loss': []}

for epoch in range(EPOCHS):
    # Training
    model.train()
    train_loss = 0

    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # Validation
    model.eval()
    val_loss = 0

    with torch.inference_mode():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)
            val_loss += loss.item()

    val_loss /= len(val_loader)

    # Save history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)

    # 3. Print losses
    print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

print("\nTraining Complete!")

In [ ]:
# TO DO
def visualize_predictions(model, dataset, num_samples=2, device='cuda'):
    model.eval()

    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4*num_samples))

    class_names = ['Background (waterbody)', 'Human divers', 'Aquatic plants and sea-grass', 'Wrecks and ruins', 'Robots (AUVs/ROVs/instruments)',"Reefs and invertebrates","Fish and vertebrates","Sea-floor and rocks"]

    with torch.inference_mode():
        for i in range(num_samples):
            image, mask = dataset[i]

            # Predict
            output = model(image.unsqueeze(0).to(device))
            pred = output.argmax(dim=1).squeeze().cpu().numpy()

            # Denormalize image
            img = image.permute(1, 2, 0).numpy()
            img = img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]
            img = np.clip(img, 0, 1)

            # Plot Image
            axes[i, 0].imshow(img)
            axes[i, 0].set_title('Image')
            axes[i, 0].axis('off')

            # Plot Ground Truth
            axes[i, 1].imshow(mask.numpy(), cmap='tab10', vmin=0, vmax=4)
            axes[i, 1].set_title('Ground Truth')
            axes[i, 1].axis('off')

            # Plot Prediction
            axes[i, 2].imshow(pred, cmap='tab10', vmin=0, vmax=4)
            axes[i, 2].set_title('Prediction')
            axes[i, 2].axis('off')

    plt.tight_layout()
    plt.show()

visualize_predictions(model, val_loader, num_samples=4, device=device)